# Dimensionality Reduction


In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('default')
import pandas as pd
import numpy as np

# Display all cell outputs
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

from IPython import get_ipython
ipython = get_ipython()

%matplotlib inline
%config InlineBackend.figure_format = 'svg'

# Set max rows and columns displayed in jupyter
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)

#the following gives access to utils folder
#where utils package stores shared code
import os
import sys
PROJECT_ROOT = os.path.abspath(os.path.join(
                  os.getcwd(),
                  os.pardir)
)

#only add it once
if (PROJECT_ROOT not in sys.path):
    sys.path.append(PROJECT_ROOT)

# autoreload extension
if 'autoreload' not in ipython.extension_manager.loaded:
    %load_ext autoreload

%autoreload 2

## The UCI wine dataset
These data are the results of a chemical analysis of wines grown in the same region in Italy but derived from three different cultivars. The analysis determined the quantities of 13 constituents found in each of the three types of wines.

In [2]:
#Let's import the data from sklearn
from sklearn.datasets import load_wine
wine=load_wine()

#Conver to pandas dataframe
df = pd.DataFrame(data=np.c_[wine['data'],wine['target']],columns=wine['feature_names']+['target'])

#Check data with info function
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 178 entries, 0 to 177
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   alcohol                       178 non-null    float64
 1   malic_acid                    178 non-null    float64
 2   ash                           178 non-null    float64
 3   alcalinity_of_ash             178 non-null    float64
 4   magnesium                     178 non-null    float64
 5   total_phenols                 178 non-null    float64
 6   flavanoids                    178 non-null    float64
 7   nonflavanoid_phenols          178 non-null    float64
 8   proanthocyanins               178 non-null    float64
 9   color_intensity               178 non-null    float64
 10  hue                           178 non-null    float64
 11  od280/od315_of_diluted_wines  178 non-null    float64
 12  proline                       178 non-null    float64
 13  targe

In [3]:
df.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0.0


## Lets see if any features are highly correlated

In [4]:
#make the heatmap fit on the screen
plt.figure(figsize=(16, 5))

# generate the complete (upper and lower) correlation matrix 
# (abs converts to absolute value, this way we only look for 1 color range)
corr = df.corr().abs()

# Generate mask for the upper triangle (see https://seaborn.pydata.org/examples/many_pairwise_correlations.html)
# the matrix is symmetric, the diagonal (all 1's) and upper triangle are visual noise, use this to mask both out
mask = np.tril(np.ones_like(corr, dtype=bool), k=-1)    #k=-1 means get rid of the diagonal

#mask out upper half 
corr = corr.where(cond=mask)

sns.heatmap(corr, annot=True);

There seems to be some correlation around total_phenols and favanoids, lets see which columns are of interest for various correlations

In [5]:
#drop extra correlated columns
def get_correlated_columns(df,correlation_threshold ):
    '''
    df: a dataframe
    correlation_threshold: select all rows and columns that have a correlation >= to this value
    return: list of tuples of form [ (col,row),...]
    '''
    # generate the correlation matrix (abs converts to absolute value, this way we only look for 1 color range)
    corr = df.corr().abs()
    # Generate mask for the upper triangle (see https://seaborn.pydata.org/examples/many_pairwise_correlations.html)
    # the matrix is symmetric, the diagonal (all 1's) and upper triangle are visual noise, use this to mask both out
    mask = np.tril(np.ones_like(corr, dtype=bool), k=-1)    #k=-1 means get rid of the diagonal
    corr = corr.where(cond=mask)
    
    correlated=[]
    for col in corr.columns:
        for i,val in enumerate(corr.loc[col]):
            if( val>= correlation_threshold):
                correlated.append((col,corr.loc[col].index[i]))
    return correlated

def drop_correlated_columns(df,correlation_threshold = .95):
    '''
    df: a dataframe
    return: df with 1 of each 2 correlated columns dropped
    '''
    correlated = get_correlated_columns(df, correlation_threshold)
    while correlated:
        print(f'dropping{correlated[0][0]}')
        df.drop(columns=[correlated[0][0]], inplace=True)
        correlated = get_correlated_columns(df, correlation_threshold)

#select all columns that meet or exceed this threshold
#thresholds are usually >.95 (95%) otherwise you risk losing some signal
#used .87 just to demo what these do
correlation_threshold = .82
correlated = get_correlated_columns(df, correlation_threshold)
print(f'Here is a list of the correlated tuples, consider dropping 1{correlated}')

Here is a list of the correlated tuples, consider dropping 1[('flavanoids', 'total_phenols'), ('target', 'flavanoids')]


In [6]:
# use a list comprehension to expand the list into tuples and then expand each tuple into a value
#then convert to a set to get rid of overlaps
cols = list(set([item for tup in correlated for item in tup]))
cols

['target', 'total_phenols', 'flavanoids']

In [7]:
#want to filter the seaborn warnings
import warnings
warnings.filterwarnings("ignore", "is_categorical_dtype")
warnings.filterwarnings("ignore", "use_inf_as_na")

In [8]:
cols=df.columns

In [9]:
#lets see what they look like   
df1=df[cols]
# sns.pairplot(df1)
# sns.pairplot(df1, hue='target')

Flavanoids and total phenals are definitely correlated, but probably not enough to get rid of either column since threshold is only 87%

## PCA for feature (dimensionality) reduction

### first scale the data since PCA requires standardization

In [10]:
import utils as ut
from sklearn.preprocessing import StandardScaler

In [11]:
dir(ut)

['PROCESSED_DATA',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 'calculate_square',
 'generate_tshirt_order',
 'hello_world',
 'my_function',
 'names',
 'np',
 'pd',
 'say_hello']

<mark>Lets remove the target, it's the type of wine, we will use it later when plotting

In [12]:
df_target=df.target
df.drop('target', axis=1, inplace=True)

In [13]:
df1=df.copy(deep=True)
df2=df.copy(deep=True)

In [14]:
list(df1.columns)

['alcohol',
 'malic_acid',
 'ash',
 'alcalinity_of_ash',
 'magnesium',
 'total_phenols',
 'flavanoids',
 'nonflavanoid_phenols',
 'proanthocyanins',
 'color_intensity',
 'hue',
 'od280/od315_of_diluted_wines',
 'proline']

In [15]:
%%time    
#comment %%time out to step into ut.scale below
ut.scale(df1, list(df1.columns))

AttributeError: module 'utils' has no attribute 'scale'

In [16]:
%%time
df2= pd.DataFrame(StandardScaler().fit_transform(df2), columns=df2.columns)

CPU times: total: 0 ns
Wall time: 5.89 ms


Sklearn standardscaler is twice as fast on this tiny dataset than the scale function in utilities<br>
In anycase we now have a dataframe to work with, lets get rid of the extra (df2) to get in the habit of conserving memory

In [17]:
del df2
import gc 
gc.collect()  #ask garbage collector to reclaim df2 memory

225

### Now PCA

In [18]:
df1.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0


In [19]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

In [20]:
PCA?

Init signature:
PCA(
    n_components=None,
    *,
    copy=True,
    whiten=False,
    svd_solver='auto',
    tol=0.0,
    iterated_power='auto',
    n_oversamples=10,
    power_iteration_normalizer='auto',
    random_state=None,
)
Docstring:     
Principal component analysis (PCA).

Linear dimensionality reduction using Singular Value Decomposition of the
data to project it to a lower dimensional space. The input data is centered
but not scaled for each feature before applying the SVD.

It uses the LAPACK implementation of the full SVD or a randomized truncated
SVD by the method of Halko et al. 2009, depending on the shape of the input
data and the number of components to extract.

With sparse inputs, the ARPACK implementation of the truncated SVD can be
used (i.e. through :func:`scipy.sparse.linalg.svds`). Alternatively, one
may consider :class:`TruncatedSVD` where the data are not centered.

Notice that this class only supports sparse inputs for some solvers such as
"arpack" and "c

In [21]:
pca = PCA(n_components=.99)

In [22]:
features_pca=pd.DataFrame(pca.fit_transform(df1))
print(f'Orig #features={df.shape[1]}, number features containing 75% of variance={features_pca.shape[1]}')

Orig #features=13, number features containing 75% of variance=1


In [23]:
features_pca.head()

,0
0,318.562979
1,303.097420
2,438.061133
3,733.240139
4,-11.571428


### Lets see what each Principle component does for us

In [24]:
#lets see how important each of those Principal Component's are
pca_ev=pca.explained_variance_ratio_
pca_ev

array([0.99809123])

In [25]:
numb_pca_feats=len(pca_ev)
fig, ax1 = plt.subplots(figsize=(16, 5))
p= sns.lineplot(y=pca_ev,x=range(0,numb_pca_feats), marker='o', ax=ax1)
sns.lineplot(y=np.cumsum(pca.explained_variance_ratio_),x=range(0,numb_pca_feats), marker='o', ax=ax1);
p.set_xlabel("Principal Component");
p.set_ylabel("Variance explained");

features_pca are the new features that define our wineset, a dataframe with the same number of rows but <mark> fewer columns

### Add the target back for plotting

In [26]:
features_pca = pd.concat([features_pca,df_target ], axis=1)
features_pca.head()

,0,target
0,318.562979,0.0
1,303.097420,0.0
2,438.061133,0.0
3,733.240139,0.0
4,-11.571428,0.0


### How much of the variance explained is in the first 2 principle components, 

In [27]:
print(f'The total variance explained by the first 2 Principle Components is {pca_ev[0]+pca_ev[1]}')

IndexError: index 1 is out of bounds for axis 0 with size 1

### Now plot using first 2 principle components

In [ ]:
# features_pca.head()
#the column names are numbers, not strings, so the x and y values below are numbers as well
features_pca.head()

In [ ]:
colors = ['#747FE3', '#8EE35D', '#E37346']
fig, ax1 = plt.subplots(figsize=(8, 8))
p= sns.scatterplot(data=features_pca, x=0, y=1, hue='target', palette=colors)

p.set_xlabel("Principal Component 0");
p.set_ylabel("Principal Component 1");

### The first 2 principle components seem to describe the data nicely

## Homework:  Visualize with t-SNE, a useful algorithm for projecting high dimensional data onto a 2 or 3 dimensonal spaces works
<mark>CAUTION: t-SNE is used just for visualization, not for dimensionality reduction since it does not preserve distance info.</mark>
See <a href="https://distill.pub/2016/misread-tsne/">How to Use t-SNE Effectively</a> for an explanation of t-SNE and it's pitfalls.
 


## BONUS:  Lets see how UMAP, another dimensionality reduction algorithm performs

See course website for UMAP intro

In [32]:
# run this once
!conda install -c conda-forge umap-learn -y

^C


In [33]:
import umap

ModuleNotFoundError: No module named 'umap'

In [29]:
#defaults to producing 2 embeddings for easy visualization
reducer = umap.UMAP()

NameError: name 'umap' is not defined

In [ ]:
# umap.UMAP?

In [ ]:
df2=df.copy()

In [ ]:
embedding = reducer.fit_transform(df2)
embedding.shape

In [ ]:
#add target back
# embeddings = pd.concat([features_pca,df_target ], axis=1)
embeddings=None


In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 8))
sns.scatterplot(x=embedding[:,0], y=embedding[:,1], hue=df_target.to_numpy(), palette=colors );